In [1]:
!pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.3.0+cu121.html
!pip install torch_geometric
!pip install deepchem
!pip install rdkit
!pip install torchinfo
!pip install molfeat

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 36.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 70.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 45.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 50.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 91.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 17.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196

In [2]:
!git clone https://github.com/AmirJlr/FDGNN.git

Cloning into 'FDGNN'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 127 (delta 73), reused 97 (delta 49), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 29.83 MiB | 11.27 MiB/s, done.
Resolving deltas: 100% (73/73), done.
Updating files: 100% (64/64), done.


In [3]:
import os
os.chdir('FDGNN')

In [4]:
!pwd

/content/FDGNN


In [5]:
!ls

assets	examples  modules    README.md	       test_results
data	models	  notebooks  requirements.txt


In [6]:
import random
import numpy as np
import torch

SEED = 37
def seed_set(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_set(SEED)

In [7]:
# %load modules/data_handler.py
import numpy as np
import pandas as pd

import torch
from torch_geometric.data import Dataset, InMemoryDataset, Data
from torch_geometric.loader import DataLoader
from torch_geometric.utils import from_smiles
from torch_geometric.utils import degree

import os
from tqdm.notebook import tqdm

import deepchem as dc

from rdkit import Chem
from rdkit.Chem import AllChem

from sklearn.model_selection import train_test_split

from molfeat.calc import FPCalculator, RDKitDescriptors2D, Pharmacophore2D, Pharmacophore3D, RDKitDescriptors3D
import datamol as dm
from molfeat.trans import MoleculeTransformer

from sklearn.decomposition import PCA

import signal

from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict



def generate_graph_list(df, smiles_column, target_column):
    graph_list = []

    for i, smile in tqdm(enumerate(df[smiles_column])):
        g = from_smiles(smile)
        g.x = g.x.float()
        y = torch.tensor(df[target_column][i], dtype=torch.float).view(1, -1)
        g.y = y
        graph_list.append(g)

    return graph_list



############################# General Loader : #############################

def load_and_process_data(dataset, splitter="random", test_size=0.1, batch_size=32):
    """
    Loads a dataset, splits it into train, validation, and test sets, and creates PyTorch Geometric data loaders.
    """
    if splitter == "random":

        data_size = len(dataset)
        train_idx, test_idx = train_test_split(list(range(data_size)), test_size=0.1)
        train_idx, valid_idx = train_test_split(train_idx, test_size = test_size)  # Split train further into train and valid

        # Create data loaders for train, validation, and test sets
        train_loader = DataLoader(dataset[train_idx], batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(dataset[valid_idx], batch_size=batch_size, shuffle=False)
        test_loader = DataLoader(dataset[test_idx], batch_size=batch_size, shuffle=False)

    else:
        raise ValueError(f"Invalid splitter type: {splitter}. Valid options are 'random' or 'scaffold'.")

    return train_loader, val_loader, test_loader



def generate_scaffold(smiles, include_chirality=False):
    """Generate the Bemis-Murcko scaffold for a given SMILES string."""
    mol = Chem.MolFromSmiles(smiles)
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(
        mol=mol, includeChirality=include_chirality)
    return scaffold


def scaffold_split_indices(smiles_list, frac_train=0.8, frac_valid=0.1, frac_test=0.1, seed=None, include_chirality=False):
    """
    Perform scaffold splitting on a list of SMILES strings and return the indices for train, validation, and test sets.

    Args:
        smiles_list (list): List of SMILES strings.
        frac_train (float): Fraction of the dataset to use for training.
        frac_valid (float): Fraction of the dataset to use for validation.
        frac_test (float): Fraction of the dataset to use for testing.
        seed (int): Random seed for shuffling the scaffolds.
        include_chirality (bool): Whether to include chirality in scaffold generation.

    Returns:
        dict: Dictionary with train, valid, and test indices as torch tensors.
    """
    np.testing.assert_almost_equal(frac_train + frac_valid + frac_test, 1.0, err_msg="The fractions must sum to 1.")

    # Set random seed for reproducibility
    rng = np.random.RandomState(seed)

    # Group SMILES by their scaffold
    scaffolds = defaultdict(list)
    for ind, smiles in enumerate(smiles_list):
        scaffold = generate_scaffold(smiles, include_chirality)
        scaffolds[scaffold].append(ind)

    # Get scaffold keys and shuffle them
    scaffold_keys = list(scaffolds.keys())
    rng.shuffle(scaffold_keys)

    # Compute the number of samples for each set
    n_total = len(smiles_list)
    n_total_valid = int(np.floor(frac_valid * n_total))
    n_total_test = int(np.floor(frac_test * n_total))

    train_index = []
    valid_index = []
    test_index = []

    # Distribute the scaffold sets into train, valid, and test sets
    for scaffold_key in scaffold_keys:
        scaffold_set = scaffolds[scaffold_key]
        if len(valid_index) + len(scaffold_set) <= n_total_valid:
            valid_index.extend(scaffold_set)
        elif len(test_index) + len(scaffold_set) <= n_total_test:
            test_index.extend(scaffold_set)
        else:
            train_index.extend(scaffold_set)

    # Return indices as torch tensors in a dictionary
    return {
        'train': torch.tensor(train_index, dtype=torch.long),
        'valid': torch.tensor(valid_index, dtype=torch.long),
        'test': torch.tensor(test_index, dtype=torch.long)
    }


class FingerprintsDescriptorsCalculator:
    def __init__(self, smiles_column):
        self.smiles_column = smiles_column

        self.valid_molecules = []
        self.valid_smiles = []
        self.invalid_indices = []

        for index, smiles in tqdm(enumerate(self.smiles_column)) :
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                print('******* Invalid Mol !!!!!!!')
                self.invalid_indices.append(index)
            else :
                self.valid_smiles.append(smiles)


        self.calc_ecfp = FPCalculator("ecfp")
        self.calc_topological = FPCalculator("topological")
        self.calc_maccs = FPCalculator("maccs")
        self.calc_estate = FPCalculator("estate")
        self.calc_rdkit2D = RDKitDescriptors2D(replace_nan=True)
        self.calc_phar2D = Pharmacophore2D()


        self.featurizer_ecfp = MoleculeTransformer(self.calc_ecfp, dtype=np.float64)
        self.featurizer_topological = MoleculeTransformer(self.calc_topological, dtype=np.float64)
        self.featurizer_maccs = MoleculeTransformer(self.calc_maccs, dtype=np.float64)
        self.featurizer_estate = MoleculeTransformer(self.calc_estate, dtype=np.float64)
        self.featurizer_rdkit2D = MoleculeTransformer(self.calc_rdkit2D, dtype=np.float64)
        self.featurizer_phar2D = MoleculeTransformer(self.calc_phar2D, dtype=np.float64)


    def calculate_ecfp(self):
        with dm.without_rdkit_log():
            return self.featurizer_ecfp(self.valid_smiles)

    def calculate_topological(self):
        with dm.without_rdkit_log():
            return self.featurizer_topological(self.valid_smiles)

    def calculate_maccs(self):
        with dm.without_rdkit_log():
            return self.featurizer_maccs(self.valid_smiles)

    def calculate_estate(self):
        with dm.without_rdkit_log():
            return self.featurizer_estate(self.valid_smiles)

    def calculate_rdkit2D(self):
        with dm.without_rdkit_log():
            return self.featurizer_rdkit2D(self.valid_smiles)

    def calculate_phar2D(self):
        with dm.without_rdkit_log():
            return self.featurizer_phar2D(self.valid_smiles)

    def get_invalid_indices(self):
        return self.invalid_indices

    def get_valid_smiles(self):
        return self.valid_smiles


# Usage Example :
# df = pd.read_csv('/content/bace.csv')
# smiles_column = df['mol'].values

# calculator = FingerprintsDescriptorsCalculator(smiles_column)

# ecfp = calculator.calculate_ecfp()
# topological = calculator.calculate_topological()
# maccs = calculator.calculate_maccs()
# estate = calculator.calculate_estate()
# rdkit2D = calculator.calculate_rdkit2D()
# phar2D = calculator.calculate_phar2D()

# phar3D = calculator.calculate_phar3D()
# rdkit3D = calculator.calculate_rdkit3D()
# invalid_indices = calculator.get_invalid_indices()


class FingerprintsDescriptorsCalculator2:
    def __init__(self, smiles_column):
        self.smiles_column = smiles_column

        self.valid_smiles = []
        self.invalid_indices = []

        for index, smiles in tqdm(enumerate(self.smiles_column)):
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                print(f'******* Invalid Mol at index {index} !!!!!!')
                self.invalid_indices.append(index)
            else:
                self.valid_smiles.append(smiles)

        self.calc_ecfp = FPCalculator("ecfp")
        self.calc_topological = FPCalculator("topological")
        self.calc_maccs = FPCalculator("maccs")
        self.calc_estate = FPCalculator("estate")
        self.calc_rdkit2D = RDKitDescriptors2D(replace_nan=True)
        self.calc_phar2D = Pharmacophore2D(replace_nan=True)

        self.featurizer_ecfp = MoleculeTransformer(self.calc_ecfp, dtype=np.float64)
        self.featurizer_topological = MoleculeTransformer(self.calc_topological, dtype=np.float64)
        self.featurizer_maccs = MoleculeTransformer(self.calc_maccs, dtype=np.float64)
        self.featurizer_estate = MoleculeTransformer(self.calc_estate, dtype=np.float64)
        self.featurizer_rdkit2D = MoleculeTransformer(self.calc_rdkit2D, dtype=np.float64)
        # self.featurizer_phar2D = MoleculeTransformer(self.calc_phar2D, dtype=np.float64)

    def calculate_phar2D(self, timeout=20):
        def timeout_handler(signum, frame):
            raise TimeoutError("Phar2D calculation timed out")

        signal.signal(signal.SIGALRM, timeout_handler)

        results = []
        remaining_smiles = []
        for index, smiles in tqdm(enumerate(self.valid_smiles)):
            signal.alarm(timeout)
            try:
                with dm.without_rdkit_log():
                    result = self.calc_phar2D(smiles)
                results.append(result)
                remaining_smiles.append(smiles)
            except TimeoutError:
                print(f"Phar2D calculation timed out for index {index}, smiles: {smiles}")
                self.invalid_indices.append(index)
            finally:
                signal.alarm(0)

        self.valid_smiles = remaining_smiles
        return np.array(results, dtype=np.float64)

    def calculate_ecfp(self):
        with dm.without_rdkit_log():
            return self.featurizer_ecfp(self.valid_smiles)

    def calculate_topological(self):
        with dm.without_rdkit_log():
            return self.featurizer_topological(self.valid_smiles)

    def calculate_maccs(self):
        with dm.without_rdkit_log():
            return self.featurizer_maccs(self.valid_smiles)

    def calculate_estate(self):
        with dm.without_rdkit_log():
            return self.featurizer_estate(self.valid_smiles)

    def calculate_rdkit2D(self):
        with dm.without_rdkit_log():
            return self.featurizer_rdkit2D(self.valid_smiles)

    def get_invalid_indices(self):
        return self.invalid_indices

    def get_valid_smiles(self):
        return self.valid_smiles


# calculator = FingerprintsDescriptorsCalculator2(smiles_column)

# phar2D = calculator.calculate_phar2D()
# ecfp = calculator.calculate_ecfp()
# topological = calculator.calculate_topological()
# maccs = calculator.calculate_maccs()
# estate = calculator.calculate_estate()
# rdkit2D = calculator.calculate_rdkit2D()
# invalid_indices = calculator.get_invalid_indices()



class PCAReducer:
    def __init__(self, n_components=64):
        self.n_components = n_components
        self.pca_ecfp = PCA(n_components=self.n_components)
        self.pca_topological = PCA(n_components=self.n_components)
        self.pca_maccs = PCA(n_components=self.n_components)
        self.pca_estate = PCA(n_components=self.n_components)
        self.pca_rdkit2D = PCA(n_components=self.n_components)
        self.pca_phar2D = PCA(n_components=self.n_components)
        # self.pca_phar3D = PCA(n_components=self.n_components)
        # self.pca_rdkit3D = PCA(n_components=self.n_components)


    def reduce_ecfp(self, ecfp_data):
        return self.pca_ecfp.fit_transform(ecfp_data)

    def reduce_topological(self, topological_data):
        return self.pca_topological.fit_transform(topological_data)

    def reduce_maccs(self, maccs_data):
        return self.pca_maccs.fit_transform(maccs_data)

    def reduce_estate(self, estate_data):
        return self.pca_estate.fit_transform(estate_data)

    def reduce_rdkit2D(self, rdkit2D_data):
        return self.pca_rdkit2D.fit_transform(rdkit2D_data)

    def reduce_phar2D(self, phar2D_data):
        return self.pca_phar2D.fit_transform(phar2D_data)

    def reduce_phar3D(self, phar3D_data):
        return self.pca_phar3D.fit_transform(phar3D_data)

    def reduce_rdkit3D(self, rdkit3D_data):
        return self.pca_rdkit3D.fit_transform(rdkit3D_data)

# Usage Example :
# N_COMPONENTS = 64
# reducer = PCAReducer(n_components=N_COMPONENTS)

# ecfp_reduced = reducer.reduce_ecfp(ecfp)
# topological_reduced = reducer.reduce_topological(topological)
# maccs_reduced = reducer.reduce_maccs(maccs)
# estate_reduced = reducer.reduce_estate(estate)
# rdkit2D_reduced = reducer.reduce_rdkit2D(rdkit2D)
# phar2D_reduced = reducer.reduce_phar2D(phar2D)

# phar3D_reduced = reducer.reduce_phar3D(phar3D)
# rdkit3D_reduced = reducer.reduce_rdkit3D(rdkit3D)


class DTsetBasic(InMemoryDataset):
    def __init__(self, root, filename, smiles_column, label_column,
                 ECFP, Topological, MACCS, EState, Rdkit2D, Phar2D):
        self.filename = filename
        self.smiles_column = smiles_column
        self.label_column = label_column

        self.ECFP = ECFP
        self.Topological = Topological
        self.MACCS = MACCS
        self.EState = EState
        self.Rdkit2D = Rdkit2D
        self.Phar2D = Phar2D

        # self.Phar3D = Phar3D
        # self.Rdkit3D = Rdkit3D

        super().__init__(root)
        self.load(self.processed_paths[0])

    @property
    def raw_file_names(self):
        return [self.filename]

    @property
    def processed_file_names(self):
        return ['data.pt']

    def download(self):
        pass  # Implement download logic if needed

    def process(self):
        # Load raw data
        data_path = os.path.join(self.raw_dir, self.filename)
        df = pd.read_csv(data_path)

        graph_list = []
        for i, smiles in tqdm(enumerate(df[self.smiles_column])):

            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                continue  # Skip invalid SMILES strings

            g = from_smiles(smiles)
            g.x = g.x.float()
            g.y = torch.tensor(df[self.label_column][i], dtype=torch.float).view(1, -1)

            g.ECFP = torch.tensor(self.ECFP[i], dtype=torch.float).view(1, -1)
            g.Topological = torch.tensor(self.Topological[i], dtype=torch.float).view(1, -1)
            g.MACCS = torch.tensor(self.MACCS[i], dtype=torch.float).view(1, -1)
            g.EState = torch.tensor(self.EState[i], dtype=torch.float).view(1, -1)
            g.Rdkit2D = torch.tensor(self.Rdkit2D[i], dtype=torch.float).view(1, -1)
            g.Phar2D = torch.tensor(self.Phar2D[i], dtype=torch.float).view(1, -1)

            # g.Phar3D = torch.tensor(self.Phar3D[i], dtype=torch.float).view(1, -1)
            # g.Rdkit3D = torch.tensor(self.Rdkit3D[i], dtype=torch.float).view(1, -1)

            graph_list.append(g)

        data_list = graph_list

        # Apply pre-filter and pre-transform
        if self.pre_filter is not None:
            data_list = [data for data in data_list if self.pre_filter(data)]

        if self.pre_transform is not None:
            data_list = [self.pre_transform(data) for data in data_list]

        # Save processed data
        self.save(data_list, self.processed_paths[0])

# dataset_64 = DTsetBasic(root='basic-64', filename='bace.csv', smiles_column='mol', label_column='Class',
#     ECFP=ecfp_reduced, Topological=topological_reduced, MACCS=maccs_reduced,
#     EState=estate_reduced, Rdkit2D=rdkit2D_reduced, Phar2D=phar2D_reduced)


class DTsetBasicMulti(InMemoryDataset):
    def __init__(self, root, filename, smiles_column,
                 ECFP, Topological, MACCS, EState, Rdkit2D, Phar2D):
        self.filename = filename
        self.smiles_column = smiles_column

        self.ECFP = ECFP
        self.Topological = Topological
        self.MACCS = MACCS
        self.EState = EState
        self.Rdkit2D = Rdkit2D
        self.Phar2D = Phar2D

        super().__init__(root)
        self.load(self.processed_paths[0])

    @property
    def raw_file_names(self):
        return [self.filename]

    @property
    def processed_file_names(self):
        return ['data.pt']

    def download(self):
        pass  # Implement download logic if needed

    def process(self):
        # Load raw data
        data_path = os.path.join(self.raw_dir, self.filename)
        df = pd.read_csv(data_path)


        graph_list = []
        for i, smiles in tqdm(enumerate(df[self.smiles_column])):

            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                continue  # Skip invalid SMILES strings

            g = from_smiles(smiles)
            g.x = g.x.float()
            g.y = torch.tensor(np.array(df.loc[i].drop(self.smiles_column).values, dtype=np.float32), dtype=torch.float).view(1, -1)

            g.ECFP = torch.tensor(self.ECFP[i], dtype=torch.float).view(1, -1)
            g.Topological = torch.tensor(self.Topological[i], dtype=torch.float).view(1, -1)
            g.MACCS = torch.tensor(self.MACCS[i], dtype=torch.float).view(1, -1)
            g.EState = torch.tensor(self.EState[i], dtype=torch.float).view(1, -1)
            g.Rdkit2D = torch.tensor(self.Rdkit2D[i], dtype=torch.float).view(1, -1)
            g.Phar2D = torch.tensor(self.Phar2D[i], dtype=torch.float).view(1, -1)

            # g.Phar3D = torch.tensor(self.Phar3D[i], dtype=torch.float).view(1, -1)
            # g.Rdkit3D = torch.tensor(self.Rdkit3D[i], dtype=torch.float).view(1, -1)

            graph_list.append(g)

        data_list = graph_list

        # Apply pre-filter and pre-transform
        if self.pre_filter is not None:
            data_list = [data for data in data_list if self.pre_filter(data)]

        if self.pre_transform is not None:
            data_list = [self.pre_transform(data) for data in data_list]

        # Save processed data
        self.save(data_list, self.processed_paths[0])

# dataset_64 = DTsetBasic(root='data/basic-64', filename='tox21_cleaned.csv', smiles_column='smiles',
#     ECFP=ecfp_reduced, Topological=topological_reduced, MACCS=maccs_reduced,
#     EState=estate_reduced, Rdkit2D=rdkit2D_reduced, Phar2D=phar2D_reduced)


class DTsetBasicExtended(Dataset):
    def __init__(self, root, filename, smiles_column, label_column,
    ECFP, Topological, MACCS, EState, MordredD, Phar2D, Phar3D, Rdkit3D):
        self.filename = filename
        self.smiles_column = smiles_column
        self.label_column = label_column

        self.ECFP = ECFP
        self.Topological = Topological
        self.MACCS = MACCS
        self.EState = EState
        self.Rdkit2D = Rdkit2D
        self.Phar2D = Phar2D
        self.Phar3D = Phar3D
        self.Rdkit3D = Rdkit3D

        super(DTsetBasicExtended, self).__init__(root)

    @property
    def raw_file_names(self):
        return [self.filename]

    @property
    def processed_file_names(self):
        data = pd.read_csv(self.raw_paths[0]).reset_index()
        return [f'data_{i}.pt' for i in data.index]

    def download(self):
        pass

    def process(self):

        data_path = os.path.join(self.raw_dir, self.filename)
        df = pd.read_csv(data_path)

        for idx, smiles in tqdm(enumerate(df[self.smiles_column])):
            mol = Chem.MolFromSmiles(smiles)

            if mol is None:
                continue  # Skip invalid SMILES strings

            node_feats = self._get_node_features(mol)
            edge_feats = self._get_edge_features(mol)
            edge_index = self._get_adjacency_info(mol)
            label = torch.tensor(df[self.label_column][idx], dtype=torch.float).view(1, -1)

            ECFP = torch.tensor(self.ECFP[idx], dtype=torch.float).view(1, -1)
            Topological = torch.tensor(self.Topological[idx], dtype=torch.float).view(1, -1)
            MACCS = torch.tensor(self.MACCS[idx], dtype=torch.float).view(1, -1)
            EState = torch.tensor(self.EState[idx], dtype=torch.float).view(1, -1)
            Rdkit2D = torch.tensor(self.Rdkit2D[idx], dtype=torch.float).view(1, -1)
            Phar2D = torch.tensor(self.Phar2D[idx], dtype=torch.float).view(1, -1)
            Phar3D = torch.tensor(self.Phar3D[idx], dtype=torch.float).view(1, -1)
            Rdkit3D = torch.tensor(self.Rdkit3D[idx], dtype=torch.float).view(1, -1)

            data = Data(
                x = node_feats,
                edge_index = edge_index,
                edge_attr = edge_feats,
                y = label,
                smiles=smiles,
                ECFP = ECFP,
                Topological = Topological,
                MACCS = MACCS,
                EState = EState,
                Rdkit2D = Rdkit2D,
                Phar2D = Phar2D,
                Phar3D = Phar3D,
                Rdkit3D = Rdkit3D
            )

           # Save processed data
            torch.save(data, os.path.join(self.processed_dir, f'data_{idx}.pt'))


    def _get_node_features(self, mol):
        """Returns a matrix of shape [Number of Nodes, Node Feature size]."""
        all_node_feats = []
        for atom in mol.GetAtoms():
            node_feats = [
                atom.GetAtomicNum(),  # Atomic number
                atom.GetDegree(),  # Degree
                atom.GetFormalCharge(),  # Formal charge
                int(atom.GetHybridization()),  # Hybridization
                atom.GetIsAromatic(),  # Aromaticity
                atom.GetTotalNumHs(),  # Total number of Hs
                atom.GetNumRadicalElectrons(),  # Radical Electrons
                atom.IsInRing(),  # In Ring
                int(atom.GetChiralTag()),  # Chirality
                atom.GetMass(),  # Atomic mass
                atom.GetExplicitValence(),  # Explicit valence
                atom.GetImplicitValence(),  # Implicit valence
                atom.GetTotalValence(),  # Total valence
                atom.GetIsotope()  # Isotope
            ]
            all_node_feats.append(node_feats)
        return torch.tensor(all_node_feats, dtype=torch.float)

    def _get_edge_features(self, mol):
        """Returns a matrix of shape [Number of edges, Edge Feature size]."""
        all_edge_feats = []
        for bond in mol.GetBonds():
            edge_feats = [
                bond.GetBondTypeAsDouble(),  # Bond type
                bond.IsInRing(),  # In Ring
                bond.GetIsAromatic(),  # Aromaticity
                int(bond.GetBondDir()),  # Bond direction
                int(bond.GetStereo()),  # Stereochemistry
                bond.GetBondLength() if hasattr(bond, 'GetBondLength') else 0  # Bond length
            ]
            # Append edge features to matrix (twice, per direction)
            all_edge_feats += [edge_feats, edge_feats]
        return torch.tensor(all_edge_feats, dtype=torch.float)

    def _get_adjacency_info(self, mol):
        """Returns adjacency information for the molecule."""
        edge_indices = []
        for bond in mol.GetBonds():
            i = bond.GetBeginAtomIdx()
            j = bond.GetEndAtomIdx()
            edge_indices += [[i, j], [j, i]]
        edge_indices = torch.tensor(edge_indices).t().to(torch.long).view(2, -1)
        return edge_indices

    def _get_labels(self, label):
        """Converts label to tensor."""
        return torch.tensor([label], dtype=torch.float)

    def len(self):
        return len(self.processed_file_names)

    def get(self, idx):
        data = torch.load(os.path.join(self.processed_dir, f'data_{idx}.pt'))
        return data


class DTsetDeepChemFeaturizer(Dataset):
    def __init__(self, root, filename, smiles_column, label_column, featurizer,
    ECFP, Topological, MACCS, EState, Rdkit2D, Phar2D, Phar3D, Rdkit3D, test=False):
        """
        root = Where the dataset should be stored. This folder is split
        into raw_dir (downloaded dataset) and processed_dir (processed data).
        """
        self.filename = filename
        self.smiles_column = smiles_column
        self.label_column = label_column
        self.featurizer = featurizer
        self.test = test

        ### prepare FP, DES :
        self.ECFP = ECFP
        self.Topological = Topological
        self.MACCS = MACCS
        self.EState = EState
        self.Rdkit2D = Rdkit2D
        self.Phar2D = Phar2D
        self.Phar3D = Phar3D
        self.Rdkit3D = Rdkit3D

        super(DTsetDeepChemFeaturizer, self).__init__(root)

    @property
    def raw_file_names(self):
        """If this file exists in raw_dir, the download is not triggered."""
        return [self.filename]

    @property
    def processed_file_names(self):
        """If these files are found in raw_dir, processing is skipped."""
        data = pd.read_csv(self.raw_paths[0]).reset_index()
        return [f'data_{i}.pt' for i in data.index]

    def download(self):
        pass  # Implement download logic if needed

    def process(self):
        # Load raw data
        data_path = os.path.join(self.raw_dir, self.filename)
        df = pd.read_csv(data_path)

        # Process each SMILES string
        for idx, smiles in tqdm(enumerate(df[self.smiles_column])):
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                continue  # Skip invalid SMILES strings

            ECFP = torch.tensor(self.ECFP[idx], dtype=torch.float).view(1, -1)
            Topological = torch.tensor(self.Topological[idx], dtype=torch.float).view(1, -1)
            MACCS = torch.tensor(self.MACCS[idx], dtype=torch.float).view(1, -1)
            EState = torch.tensor(self.EState[idx], dtype=torch.float).view(1, -1)
            Rdkit2D = torch.tensor(self.Rdkit2D[idx], dtype=torch.float).view(1, -1)
            Phar2D = torch.tensor(self.Phar2D[idx], dtype=torch.float).view(1, -1)
            Phar3D = torch.tensor(self.Phar3D[idx], dtype=torch.float).view(1, -1)
            Rdkit3D = torch.tensor(self.Rdkit3D[idx], dtype=torch.float).view(1, -1)

            # Featurize molecule
            f = self.featurizer._featurize(mol)
            # To pyg
            data = f.to_pyg_graph()
            data.y = torch.tensor(df[self.label_column][idx], dtype=torch.float).view(1, -1)
            data.smiles = smiles

            data.ECFP = ECFP,
            data.Topological = Topological,
            data.MACCS = MACCS,
            data.EState = EState,
            data.Rdkit2D = Rdkit2D,
            data.Phar2D = Phar2D,
            data.Phar3D = Phar3D,
            data.Rdkit3D = Rdkit3D

            # Save processed data
            torch.save(data, os.path.join(self.processed_dir, f'data_{idx}.pt'))

    def len(self):
        return len(self.processed_file_names)

    def get(self, idx):
        data = torch.load(os.path.join(self.processed_dir, f'data_{idx}.pt'))
        return data


# Featurizer
# featurizer = dc.feat.MolGraphConvFeaturizer(use_edges=True)



class DTsetMolGraphConvFeaturizer(Dataset):
    def __init__(self, root, filename, smiles_column, label_column, test=False, transform=None, pre_transform=None):
        """
        root = Where the dataset should be stored. This folder is split
        into raw_dir (downloaded dataset) and processed_dir (processed data).
        """
        self.filename = filename
        self.smiles_column = smiles_column
        self.label_column = label_column
        self.test = test
        super(DTsetMolGraphConvFeaturizer, self).__init__(root, transform, pre_transform)

    @property
    def raw_file_names(self):
        """If this file exists in raw_dir, the download is not triggered."""
        return [self.filename]

    @property
    def processed_file_names(self):
        """If these files are found in raw_dir, processing is skipped."""
        data = pd.read_csv(self.raw_paths[0]).reset_index()
        return [f'data_{i}.pt' for i in data.index]

    def download(self):
        pass  # Implement download logic if needed

    def process(self):
        # Load raw data
        data_path = os.path.join(self.raw_dir, self.filename)
        df = pd.read_csv(data_path)

        # Featurizer
        featurizer = dc.feat.MolGraphConvFeaturizer(use_edges=True)

        # Process each SMILES string
        for idx, smiles in tqdm(enumerate(df[self.smiles_column])):
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                continue  # Skip invalid SMILES strings

            # Featurize molecule
            f = featurizer._featurize(mol)
            data = f.to_pyg_graph()
            data.y = torch.tensor(df[self.label_column][idx], dtype=torch.float).view(1, -1)
            data.smiles = smiles

            # Save processed data
            torch.save(data, os.path.join(self.processed_dir, f'data_{idx}.pt'))

    def len(self):
        return len(self.processed_file_names)

    def get(self, idx):
        data = torch.load(os.path.join(self.processed_dir, f'data_{idx}.pt'))
        return data


wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.
Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead
wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.


In [8]:
from modules.data_handler import scaffold_split_indices, FingerprintsDescriptorsCalculator, PCAReducer, DTsetBasic

In [9]:
import pandas as pd

df = pd.read_csv('/content/FDGNN/data/datasets/BBBP.csv')
smiles_column = df['smiles'].values

In [10]:
len(smiles_column)

2050

In [11]:
calculator = FingerprintsDescriptorsCalculator(smiles_column)

ecfp = calculator.calculate_ecfp()
topological = calculator.calculate_topological()
maccs = calculator.calculate_maccs()
estate = calculator.calculate_estate()
rdkit2D = calculator.calculate_rdkit2D()
phar2D = calculator.calculate_phar2D()

invalid_indices = calculator.get_invalid_indices()
valid_smiles = calculator.get_valid_smiles()

0it [00:00, ?it/s]

[13:51:54] Explicit valence for atom # 1 N, 4, is greater than permitted
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] Explicit valence for atom # 6 N, 4, is greater than permitted
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] Explicit valence for atom # 6 N, 4, is greater than permitted
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] Explicit valence for atom # 11 N, 4, is greater than pe

******* Invalid Mol !!!!!!!
******* Invalid Mol !!!!!!!
******* Invalid Mol !!!!!!!
******* Invalid Mol !!!!!!!
******* Invalid Mol !!!!!!!
******* Invalid Mol !!!!!!!
******* Invalid Mol !!!!!!!
******* Invalid Mol !!!!!!!
******* Invalid Mol !!!!!!!
******* Invalid Mol !!!!!!!
******* Invalid Mol !!!!!!!


[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not removing hydrogen atom without neighbors
[13:51:54] WARNING: not r

In [12]:
# Usage Example :
N_COMPONENTS = 64
reducer = PCAReducer(n_components=N_COMPONENTS)

ecfp_reduced = reducer.reduce_ecfp(ecfp)
topological_reduced = reducer.reduce_topological(topological)
maccs_reduced = reducer.reduce_maccs(maccs)
estate_reduced = reducer.reduce_estate(estate)
rdkit2D_reduced = reducer.reduce_rdkit2D(rdkit2D)
phar2D_reduced = reducer.reduce_phar2D(phar2D)

# phar3D_reduced = reducer.reduce_phar3D(phar3D)
# rdkit3D_reduced = reducer.reduce_rdkit3D(rdkit3D)

In [13]:
directory = 'data/raw'
CSV_PATH = 'data/raw/BBBP_cleaned.csv'

if not os.path.exists(directory):
    os.makedirs(directory)

df.drop(invalid_indices).to_csv(CSV_PATH)

In [14]:
dataset_64 = DTsetBasic(root='data', filename='BBBP_cleaned.csv', smiles_column='smiles', label_column='p_np',
    ECFP=ecfp_reduced, Topological=topological_reduced, MACCS=maccs_reduced,
    EState=estate_reduced, Rdkit2D=rdkit2D_reduced, Phar2D=phar2D_reduced)
# ,Phar3D=phar3D_reduced, Rdkit3D=rdkit3D_reduced

Processing...


0it [00:00, ?it/s]

Done!


In [15]:
dataset_64[0]

Data(x=[20, 9], edge_index=[2, 40], edge_attr=[40, 3], smiles='[Cl].CC(C)NCC(O)COc1cccc2ccccc12', y=[1, 1], ECFP=[1, 64], Topological=[1, 64], MACCS=[1, 64], EState=[1, 64], Rdkit2D=[1, 64], Phar2D=[1, 64])

In [16]:
# from modules.data_handler import load_and_process_data
# train_loader_DTsetBasic, valid_loader_DTsetBasic, test_loader_DTsetBasic = load_and_process_data(dataset_64, test_size=0.2)

In [17]:
### Scaffold Splitting

split_idx = scaffold_split_indices(valid_smiles, seed=SEED)
train_loader = DataLoader(dataset_64[split_idx["train"]], batch_size=32, shuffle=True)
valid_loader = DataLoader(dataset_64[split_idx["valid"]], batch_size=32, shuffle=False)
test_loader  = DataLoader(dataset_64[split_idx["test"]], batch_size=32, shuffle=False)

In [18]:
# %load modules/utils_classification.py
import os
import numpy as np
import torch
import torch.nn as nn
from torch import device
from torch.utils.data import DataLoader
from torch.nn import Linear
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

from torch_geometric.nn import GINConv
from torch_geometric.nn import global_add_pool
from torch_geometric.loader import DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from torch.optim import Adam


from torch_geometric.nn import GCNConv, TopKPooling, global_mean_pool
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp

from copy import deepcopy
from math import sqrt
from tqdm.notebook import tqdm


def run_epoch_cls(model, optimizer, data_loader, loss_function, device, edge_attr, pass_data):
    """
    Runs a single training epoch for a PyG model on a graph property prediction task.

    Args:
        model (torch.nn.Module): The PyG model to be trained.
        optimizer (torch.optim.Optimizer, optional): The optimizer for training. Defaults to None.
        data_loader (torch_geometric.data.DataLoader): The data loader for the training data.
        loss_function (torch.nn.Module, optional): The loss function to use. Defaults to BCEWithLogitsLoss().
        device (str, optional): The device to use for training ("cpu" or "cuda"). Defaults to "cpu".

    Returns:
        tuple: A tuple containing the average loss and ROC-AUC score for the epoch.
    """

    model.to(device)
    model.train() if optimizer is not None else model.eval()

    y_true = []
    y_pred = []
    losses = []

    for step, data in enumerate(tqdm(data_loader, desc="Iteration")):  # Iterate in batches over the training dataset.
        data = data.to(device)  # Move data batch to device

        if edge_attr :
            if pass_data :
                pred = model(data.x, data.edge_index, data.edge_attr, data.batch, data)
            else :
                pred = model(data.x, data.edge_index, data.edge_attr, data.batch)
        else :
            if pass_data :
                pred = model(data.x, data.edge_index, data.batch, data)
            else :
                pred = model(data.x, data.edge_index, data.batch)

        loss = loss_function(pred, data.y.to(torch.float32))  # Calculate loss

        if optimizer is not None:
            optimizer.zero_grad()  # Clear gradients
            loss.backward()  # Backpropagation
            optimizer.step()  # Update model parameters

        losses.append(loss.detach().cpu().numpy())
        y_true.append(data.y.view(pred.shape).detach().cpu())
        y_pred.append(pred.detach().cpu())

    y_true = torch.cat(y_true, dim=0).numpy()
    y_pred = torch.cat(y_pred, dim=0).numpy()

    # Calculate ROC-AUC score using sklearn
    auc_roc = roc_auc_score(y_true, y_pred)

    return np.array(losses).mean(), auc_roc



def train_cls(model, optimizer, loss_function, train_loader, val_loader, num_epochs, device, edge_attr, pass_data, tensorboard_writer):

    writer = SummaryWriter(f'runs/{tensorboard_writer}')

    best_model = None
    best_val_auc = 0
    best_val_loss = float('inf')

    for epoch in range(1, num_epochs + 1):
        train_loss, train_auc = run_epoch_cls(model, optimizer, train_loader,loss_function, device, edge_attr, pass_data)
        writer.add_scalar('loss/train', train_loss, epoch)
        writer.add_scalar('auc/train', train_auc, epoch)

        val_loss, val_auc = run_epoch_cls(model, None, val_loader,loss_function, device, edge_attr, pass_data)
        writer.add_scalar('loss/val', val_loss, epoch)
        writer.add_scalar('auc/val', val_auc, epoch)

        print(f'Epoch: {epoch:03d}, Train loss: {train_loss:.4f}, Train ROC-AUC: {train_auc:.4f}, Val loss: {val_loss:.4f}, Val ROC-AUC: {val_auc:.4f}')

        if val_auc > best_val_auc :
            best_val_auc = val_auc
            best_val_loss = val_loss
            best_model = deepcopy(model)

    writer.close()  # Close TensorBoard writer
    return {
        'best_model': best_model,
        'best_val_loss': best_val_loss,
        'best_val_auc': best_val_auc
    }


# results = train_cls(model, optimizer, loss_function, train_loader, val_loader, num_epochs, device, edge_attr, pass_data, tensorboard_writer)
# best_model = results['best_model']
# best_val_rmse = results['best_val_rmse']

# # Save the best model
# torch.save(best_model.state_dict(), 'best_model.pth')

# # To load the model later
# # Instantiate the model class first (ensure the model class is defined the same way)
# model = YourModelClass()
# model.load_state_dict(torch.load('best_model.pth'))
# model.to(device)



######### Multi Task Classification #########

def multi_task_loss(pred, target, loss_function):
    mask = ~torch.isnan(target)
    if mask.any():
        loss = loss_function(pred[mask], target[mask].to(torch.float32))
        return loss.mean()  # Reduce the loss across the batch
    return torch.tensor(0.0, requires_grad=True)


def run_epoch_multi_cls(model, optimizer, data_loader, loss_function, device, edge_attr, pass_data):
    model.to(device)
    model.train() if optimizer is not None else model.eval()

    y_true = []
    y_pred = []
    losses = []

    for step, data in enumerate(tqdm(data_loader, desc="Iteration")):
        data = data.to(device)

        if edge_attr:
            if pass_data:
                pred = model(data.x, data.edge_index, data.edge_attr, data.batch, data)
            else:
                pred = model(data.x, data.edge_index, data.edge_attr, data.batch)
        else:
            if pass_data:
                pred = model(data.x, data.edge_index, data.batch, data)
            else:
                pred = model(data.x, data.edge_index, data.batch)

        loss = multi_task_loss(pred, data.y, loss_function)

        if optimizer is not None:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        losses.append(loss.detach().cpu().numpy())
        y_true.append(data.y.view(pred.shape).detach().cpu())
        y_pred.append(pred.detach().cpu())

    y_true = torch.cat(y_true, dim=0).numpy()
    y_pred = torch.cat(y_pred, dim=0).numpy()

    # Calculate ROC-AUC score for each task using sklearn
    auc_roc = []
    for i in range(y_true.shape[1]):
        valid_indices = ~np.isnan(y_true[:, i])
        if valid_indices.any():
            try:
                auc_roc.append(roc_auc_score(y_true[valid_indices, i], y_pred[valid_indices, i]))
            except ValueError as e:
                print(f"Error calculating ROC AUC for task {i}: {e}")
                auc_roc.append(np.nan)
        else:
            auc_roc.append(np.nan)

    avg_auc_roc = np.nanmean(auc_roc)

    return np.array(losses).mean(), avg_auc_roc



def train_multi_cls(model, optimizer, loss_function, train_loader, val_loader, num_epochs, device, edge_attr, pass_data, tensorboard_writer):

    writer = SummaryWriter(f'runs/{tensorboard_writer}')

    best_model = None
    best_val_auc = 0
    best_val_loss = float('inf')

    for epoch in range(1, num_epochs + 1):
        train_loss, train_auc = run_epoch_multi_cls(model, optimizer, train_loader,loss_function, device, edge_attr, pass_data)
        writer.add_scalar('loss/train', train_loss, epoch)
        writer.add_scalar('auc/train', train_auc, epoch)

        val_loss, val_auc = run_epoch_multi_cls(model, None, val_loader,loss_function, device, edge_attr, pass_data)
        writer.add_scalar('loss/val', val_loss, epoch)
        writer.add_scalar('auc/val', val_auc, epoch)

        print(f'Epoch: {epoch:03d}, Train loss: {train_loss:.4f}, Train ROC-AUC: {train_auc:.4f}, Val loss: {val_loss:.4f}, Val ROC-AUC: {val_auc:.4f}')

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_val_loss = val_loss
            best_model = deepcopy(model)

    writer.close()  # Close TensorBoard writer

    return {
        'best_model': best_model,
        'best_val_loss': best_val_loss,
        'best_val_auc': best_val_auc
    }

In [19]:
from modules.utils_classification import run_epoch_cls, train_cls

In [20]:
# %load models/model_poolings2.py
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import (
    GATv2Conv, GINEConv, BatchNorm,
    global_mean_pool, global_max_pool, global_add_pool, GlobalAttention
)
from torch_geometric.data import Data, Batch

############### LSTM Pooling ###############
class LSTMAttentionPooling(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, x, batch):
        num_graphs = batch.max().item() + 1
        pooled_outputs = []
        for i in range(num_graphs):
            node_embeds = x[batch == i].unsqueeze(0)  # [1, num_nodes_in_graph, input_dim]
            h_0 = torch.zeros(self.lstm.num_layers, 1, self.lstm.hidden_size, device=x.device)
            c_0 = torch.zeros(self.lstm.num_layers, 1, self.lstm.hidden_size, device=x.device)
            lstm_out, _ = self.lstm(node_embeds, (h_0, c_0)) # [1, num_nodes_in_graph, hidden_dim]
            attention_weights = F.softmax(self.attention(lstm_out.squeeze(0)), dim=0) # [num_nodes_in_graph, 1]
            graph_embedding = torch.sum(attention_weights * lstm_out.squeeze(0), dim=0) # [hidden_dim]
            pooled_outputs.append(graph_embedding)
        return torch.stack(pooled_outputs, dim=0) # [num_graphs, hidden_dim]


############### GRU Pooling ###############
class GRUAttentionPooling(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, x, batch):
        pooled_outputs = []
        num_graphs = batch.max().item() + 1
        for i in range(num_graphs):
            nodes_in_graph = x[batch == i].unsqueeze(0) # [1, num_nodes_in_graph, input_dim]
            h_0 = torch.zeros(self.gru.num_layers, 1, self.gru.hidden_size, device=x.device)
            gru_out, _ = self.gru(nodes_in_graph, h_0) # [1, num_nodes_in_graph, hidden_dim]
            attention_weights = F.softmax(self.attention(gru_out.squeeze(0)), dim=0) # [num_nodes_in_graph, 1]
            graph_embedding = torch.sum(attention_weights * gru_out.squeeze(0), dim=0) # [hidden_dim]
            pooled_outputs.append(graph_embedding)
        return torch.stack(pooled_outputs, dim=0) # [num_graphs, hidden_dim]


############### Main Model (GINGAT) ###############
class GINGAT(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_channels, out_channels, heads,
                 dropout, pooling_type, num_tasks, use_dummy=True, feature_mode="both"):
        """
        Args:
            feature_mode (str): Controls which handcrafted features to use.
                Options: "fps", "descs", "both".
        """
        super().__init__()
        self.use_dummy = use_dummy
        self.pooling_type = pooling_type
        self.feature_mode = feature_mode

        self.out_channels = out_channels
        self.hidden_channels = hidden_channels

        # === Graph backbone (GINEConv layers) ===
        self.graph_conv1 = GINEConv(nn.Sequential(
            nn.Linear(node_dim, hidden_channels), nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels)
        ), edge_dim=edge_dim)
        self.graph_bn1 = BatchNorm(hidden_channels)

        self.graph_conv2 = GINEConv(nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels), nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels)
        ), edge_dim=edge_dim)
        self.graph_bn2 = BatchNorm(hidden_channels)

        self.graph_conv3 = GINEConv(nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels), nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels)
        ), edge_dim=edge_dim)
        self.graph_bn3 = BatchNorm(hidden_channels)

        self.graph_conv4 = GINEConv(nn.Sequential(
            nn.Linear(hidden_channels, out_channels), nn.ReLU(),
            nn.Linear(out_channels, out_channels)
        ), edge_dim=edge_dim)
        self.graph_bn4 = BatchNorm(out_channels)

        # === Graph Pooling Layer ===
        if pooling_type == 'lstm':
            self.pooling = LSTMAttentionPooling(out_channels, out_channels)
        elif pooling_type == 'gru':
            self.pooling = GRUAttentionPooling(out_channels, out_channels)
        elif pooling_type == 'attention':
            self.pooling = GlobalAttention(gate_nn=nn.Linear(out_channels, 1))
        elif pooling_type == 'mean':
            self.pooling = global_mean_pool
        elif pooling_type == 'max':
            self.pooling = global_max_pool
        elif pooling_type == 'sum':
            self.pooling = global_add_pool
        else:
            raise ValueError("Pooling must be one of 'lstm', 'gru', 'attention', 'mean', 'max', 'sum'")

        # === Dummy graph branch ===
        if self.use_dummy:
            self.node_conv1 = GATv2Conv(out_channels, hidden_channels, heads=heads, concat=False)
            self.node_bn1 = BatchNorm(hidden_channels)
            self.residual_proj = nn.Linear(out_channels, hidden_channels)
        else:
            self.node_conv1 = None
            self.node_bn1 = None
            self.residual_proj = None
            self.ablation_proj = None  # Will be initialized dynamically

        # === Output head ===
        self.fc1 = nn.Linear(hidden_channels, hidden_channels // 2)
        self.fc2 = nn.Linear(hidden_channels // 2, num_tasks)
        self.dropout = nn.Dropout(dropout)

        self.last_attention = None
        self.reset_parameters()

    def forward(self, x, edge_index, edge_attr, batch, data):
        device = x.device
        edge_attr = edge_attr.float().to(device)

        # === GNN Encoder ===
        x = self.graph_conv1(x, edge_index, edge_attr)
        x = self.graph_bn1(x); x = F.relu(x); x = self.dropout(x)
        x = self.graph_conv2(x, edge_index, edge_attr)
        x = self.graph_bn2(x); x = F.relu(x); x = self.dropout(x)
        x = self.graph_conv3(x, edge_index, edge_attr)
        x = self.graph_bn3(x); x = F.relu(x); x = self.dropout(x)
        x = self.graph_conv4(x, edge_index, edge_attr)
        x = self.graph_bn4(x); x = F.relu(x)  # [num_nodes_in_batch, out_channels]

        # === Graph Pooling ===
        graph_out = self.pooling(x, batch)  # [batch_size, out_channels]

        # === Feature Selection based on `feature_mode` ===
        if self.feature_mode == "fps":
            selected_features = [
                data.ECFP.to(device),
                data.Topological.to(device),
                data.MACCS.to(device),
                data.EState.to(device)
            ]
        elif self.feature_mode == "descs":
            selected_features = [
                data.Rdkit2D.to(device),
                data.Phar2D.to(device)
            ]
        elif self.feature_mode == "both":
            selected_features = [
                data.ECFP.to(device),
                data.Topological.to(device),
                data.MACCS.to(device),
                data.EState.to(device),
                data.Rdkit2D.to(device),
                data.Phar2D.to(device)
            ]
        else:
            raise ValueError(f"Invalid feature_mode: {self.feature_mode}. Choose from 'fingerprints', 'descriptors', 'both'.")

        # Ensure all features are 2D: [batch_size, feature_dim]
        features_2d = []
        for f in selected_features:
            if f.dim() == 1:
                features_2d.append(f.unsqueeze(1))  # [batch_size, 1]
            else:
                features_2d.append(f.view(graph_out.size(0), -1))  # [batch_size, N]

        # === Apply Layer Normalization ===
        graph_out = F.layer_norm(graph_out, graph_out.size()[1:])
        normalized_features = [F.layer_norm(f, f.size()[1:]) for f in features_2d]

        if self.use_dummy:
            # === Process via Dummy Graph & GAT ===
            dummy_graphs = []
            for i in range(graph_out.size(0)):
                dummy_graph = self.create_dummy_graph(
                    graph_out[i].unsqueeze(0),
                    [f[i].unsqueeze(0) for f in normalized_features],
                    device
                )
                dummy_graphs.append(dummy_graph)

            batched_dummy = Batch.from_data_list(dummy_graphs).to(device)
            x_dummy, edge_index_dummy = batched_dummy.x, batched_dummy.edge_index

            out1 = self.node_conv1(x_dummy, edge_index_dummy, return_attention_weights=True)
            if isinstance(out1, tuple):
                x_node, (attn_edge_index, attn_alpha) = out1
            else:
                x_node, attn_edge_index, attn_alpha = out1, None, None

            x_node = self.node_bn1(x_node); x_node = F.relu(x_node)

            self.last_attention = {
                "edge_index": attn_edge_index.detach().cpu() if attn_edge_index is not None else None,
                "alpha": attn_alpha.detach().cpu() if attn_alpha is not None else None
            }

            # Extract central (target) node embeddings
            num_feats_per_graph = len(normalized_features)
            stride = num_feats_per_graph + 1
            central_indices = torch.arange(0, len(dummy_graphs) * stride, stride, device=device)
            processed_central = x_node[central_indices]  # [batch_size, hidden_channels]

            # Add Residual Connection
            projected_central = self.residual_proj(graph_out)  # [batch_size, hidden_channels]
            x_processed = processed_central + projected_central  # [batch_size, hidden_channels]

        else:
            # === Ablation: Direct Concatenation ===
            feat_cat = torch.cat([graph_out] + normalized_features, dim=1)  # [batch_size, total_dim]

            if self.ablation_proj is None:
                total_concat_dim = feat_cat.size(1)
                self.ablation_proj = nn.Linear(total_concat_dim, self.hidden_channels).to(device)

            x_processed = F.relu(self.ablation_proj(feat_cat))  # [batch_size, hidden_channels]
            self.last_attention = None

        # === Final Prediction Head ===
        x_final = F.relu(self.fc1(x_processed))
        x_final = self.dropout(x_final)
        return self.fc2(x_final)

    def create_dummy_graph(self, graph_embedding, features, device):
        """
        Creates a dummy graph Data object.
        Args:
            graph_embedding: Tensor of shape [1, out_channels] for the central node.
            features: List of tensors, each of shape [1, feature_dim] for peripheral nodes.
        Returns:
            PyG Data object.
        """
        node_features = torch.cat([graph_embedding] + features, dim=0)  # [1 + N, D]
        edges = [[0, i] for i in range(1, len(features) + 1)]  # Central node (0) -> all features
        edge_index = torch.tensor(edges, dtype=torch.long, device=device).t().contiguous()
        return Data(x=node_features, edge_index=edge_index)

    def reset_parameters(self):
        # Reset GNN backbone
        for conv, bn in [
            (self.graph_conv1, self.graph_bn1),
            (self.graph_conv2, self.graph_bn2),
            (self.graph_conv3, self.graph_bn3),
            (self.graph_conv4, self.graph_bn4)
        ]:
            conv.reset_parameters()
            bn.reset_parameters()

        # Reset pooling
        if hasattr(self.pooling, 'reset_parameters'):
            self.pooling.reset_parameters()
        elif self.pooling_type == 'lstm':
            self.pooling.lstm.reset_parameters()
            self.pooling.attention.reset_parameters()
        elif self.pooling_type == 'gru':
            self.pooling.gru.reset_parameters()
            self.pooling.attention.reset_parameters()

        # Reset dummy graph components
        if self.use_dummy:
            if self.node_conv1 is not None:
                self.node_conv1.reset_parameters()
            if self.node_bn1 is not None:
                self.node_bn1.reset_parameters()
            if self.residual_proj is not None:
                self.residual_proj.reset_parameters()
        else:
            if self.ablation_proj is not None:
                self.ablation_proj.reset_parameters()

        # Reset final layers
        self.fc1.reset_parameters()
        self.fc2.reset_parameters()

In [21]:
from models.model_poolings2 import GINGAT

In [22]:
### Configs

import torch
from torchinfo import summary

EPOCHS = 50
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LOSS_FUNCTION = torch.nn.BCEWithLogitsLoss()

## Compare Models

- ### model_gru_dummy_both

In [25]:
model_gru_dummy_both = GINGAT(node_dim=9,
                              edge_dim=3,
                              hidden_channels=64,
                              out_channels=N_COMPONENTS,
                              heads=6, dropout=0.2,
                              pooling_type='gru',
                              num_tasks=1,
                              use_dummy=True,
                              feature_mode='both')

optimizer_gru_dummy_both = torch.optim.Adam(model_gru_dummy_both.parameters(), lr=0.001, weight_decay=0.0001)

summary(model_gru_dummy_both)

Layer (type:depth-idx)                   Param #
GINGAT                                   --
├─GINEConv: 1-1                          --
│    └─SumAggregation: 2-1               --
│    └─Sequential: 2-2                   --
│    │    └─Linear: 3-1                  640
│    │    └─ReLU: 3-2                    --
│    │    └─Linear: 3-3                  4,160
│    └─Linear: 2-3                       36
├─BatchNorm: 1-2                         --
│    └─BatchNorm1d: 2-4                  128
├─GINEConv: 1-3                          --
│    └─SumAggregation: 2-5               --
│    └─Sequential: 2-6                   --
│    │    └─Linear: 3-4                  4,160
│    │    └─ReLU: 3-5                    --
│    │    └─Linear: 3-6                  4,160
│    └─Linear: 2-7                       256
├─BatchNorm: 1-4                         --
│    └─BatchNorm1d: 2-8                  128
├─GINEConv: 1-5                          --
│    └─SumAggregation: 2-9               --
│    └─Sequent

In [26]:
results_gru_dummy_both = train_cls(model = model_gru_dummy_both,
    optimizer = optimizer_gru_dummy_both,
    loss_function = LOSS_FUNCTION,
    train_loader = train_loader,
    val_loader = valid_loader,
    num_epochs = EPOCHS,
    device = device,
    edge_attr = True,
    pass_data = True,
    tensorboard_writer = "model_gru_dummy_both")

Iteration:   0%|          | 0/52 [00:00<?, ?it/s]

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch: 001, Train loss: 0.5138, Train ROC-AUC: 0.6778, Val loss: 0.3413, Val ROC-AUC: 0.9169


Iteration:   0%|          | 0/52 [00:00<?, ?it/s]

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

...

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch: 049, Train loss: 0.1802, Train ROC-AUC: 0.9723, Val loss: 0.2815, Val ROC-AUC: 0.9475


Iteration:   0%|          | 0/52 [00:00<?, ?it/s]

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch: 050, Train loss: 0.1746, Train ROC-AUC: 0.9736, Val loss: 0.3012, Val ROC-AUC: 0.9368


- ### model_mean_dummy_both

In [27]:
model_mean_dummy_both = GINGAT(node_dim=9,
                               edge_dim=3,
                               hidden_channels=96,
                               out_channels=N_COMPONENTS,
                               heads=6, dropout=0.2,
                               pooling_type='mean',
                               num_tasks=1,
                               use_dummy=True,
                               feature_mode='both')

optimizer_mean_dummy_both = torch.optim.Adam(model_mean_dummy_both.parameters(), lr=0.001, weight_decay=0.0001)

summary(model_mean_dummy_both)

Layer (type:depth-idx)                   Param #
GINGAT                                   --
├─GINEConv: 1-1                          --
│    └─SumAggregation: 2-1               --
│    └─Sequential: 2-2                   --
│    │    └─Linear: 3-1                  960
│    │    └─ReLU: 3-2                    --
│    │    └─Linear: 3-3                  9,312
│    └─Linear: 2-3                       36
├─BatchNorm: 1-2                         --
│    └─BatchNorm1d: 2-4                  192
├─GINEConv: 1-3                          --
│    └─SumAggregation: 2-5               --
│    └─Sequential: 2-6                   --
│    │    └─Linear: 3-4                  9,312
│    │    └─ReLU: 3-5                    --
│    │    └─Linear: 3-6                  9,312
│    └─Linear: 2-7                       384
├─BatchNorm: 1-4                         --
│    └─BatchNorm1d: 2-8                  192
├─GINEConv: 1-5                          --
│    └─SumAggregation: 2-9               --
│    └─Sequent

In [28]:
results_mean_dummy_both = train_cls(model = model_mean_dummy_both,
    optimizer = optimizer_mean_dummy_both,
    loss_function = LOSS_FUNCTION,
    train_loader = train_loader,
    val_loader = valid_loader,
    num_epochs = EPOCHS,
    device = device,
    edge_attr = True,
    pass_data = True,
    tensorboard_writer = "model_mean_dummy_both")

Iteration:   0%|          | 0/52 [00:00<?, ?it/s]

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch: 001, Train loss: 0.5025, Train ROC-AUC: 0.7092, Val loss: 0.4150, Val ROC-AUC: 0.8156


Iteration:   0%|          | 0/52 [00:00<?, ?it/s]

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

...

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch: 049, Train loss: 0.3198, Train ROC-AUC: 0.8982, Val loss: 0.3247, Val ROC-AUC: 0.9159


Iteration:   0%|          | 0/52 [00:00<?, ?it/s]

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch: 050, Train loss: 0.3124, Train ROC-AUC: 0.9068, Val loss: 0.3325, Val ROC-AUC: 0.9147


- ### model_gru_dummy_fp

In [29]:
model_gru_dummy_fps = GINGAT(node_dim=9,
                            edge_dim=3,
                            hidden_channels=96,
                            out_channels=N_COMPONENTS,
                            heads=6, dropout=0.2,
                            pooling_type='gru',
                            num_tasks=1,
                            use_dummy=True,
                            feature_mode='fps')

optimizer_gru_dummy_fps = torch.optim.Adam(model_gru_dummy_fps.parameters(), lr=0.001, weight_decay=0.0001)

summary(model_gru_dummy_fps)

Layer (type:depth-idx)                   Param #
GINGAT                                   --
├─GINEConv: 1-1                          --
│    └─SumAggregation: 2-1               --
│    └─Sequential: 2-2                   --
│    │    └─Linear: 3-1                  960
│    │    └─ReLU: 3-2                    --
│    │    └─Linear: 3-3                  9,312
│    └─Linear: 2-3                       36
├─BatchNorm: 1-2                         --
│    └─BatchNorm1d: 2-4                  192
├─GINEConv: 1-3                          --
│    └─SumAggregation: 2-5               --
│    └─Sequential: 2-6                   --
│    │    └─Linear: 3-4                  9,312
│    │    └─ReLU: 3-5                    --
│    │    └─Linear: 3-6                  9,312
│    └─Linear: 2-7                       384
├─BatchNorm: 1-4                         --
│    └─BatchNorm1d: 2-8                  192
├─GINEConv: 1-5                          --
│    └─SumAggregation: 2-9               --
│    └─Sequent

In [30]:
results_gru_dummy_fps = train_cls(model = model_gru_dummy_fps,
    optimizer = optimizer_gru_dummy_fps,
    loss_function = LOSS_FUNCTION,
    train_loader = train_loader,
    val_loader = valid_loader,
    num_epochs = EPOCHS,
    device = device,
    edge_attr = True,
    pass_data = True,
    tensorboard_writer = "model_gru_dummy_fps")

Iteration:   0%|          | 0/52 [00:00<?, ?it/s]

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch: 001, Train loss: 0.5248, Train ROC-AUC: 0.6493, Val loss: 0.4951, Val ROC-AUC: 0.8357


Iteration:   0%|          | 0/52 [00:00<?, ?it/s]

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

...

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch: 049, Train loss: 0.1748, Train ROC-AUC: 0.9776, Val loss: 0.2962, Val ROC-AUC: 0.9494


Iteration:   0%|          | 0/52 [00:00<?, ?it/s]

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch: 050, Train loss: 0.1603, Train ROC-AUC: 0.9779, Val loss: 0.3745, Val ROC-AUC: 0.9330


- ### model_gru_dummy_desc

In [31]:
model_gru_dummy_descs = GINGAT(node_dim=9,
                               edge_dim=3,
                               hidden_channels=96,
                               out_channels=N_COMPONENTS,
                               heads=6, dropout=0.2,
                               pooling_type='gru',
                               num_tasks=1,
                               use_dummy=True,
                               feature_mode='descs')

optimizer_gru_dummy_descs = torch.optim.Adam(model_gru_dummy_descs.parameters(), lr=0.001, weight_decay=0.0001)

summary(model_gru_dummy_descs)

Layer (type:depth-idx)                   Param #
GINGAT                                   --
├─GINEConv: 1-1                          --
│    └─SumAggregation: 2-1               --
│    └─Sequential: 2-2                   --
│    │    └─Linear: 3-1                  960
│    │    └─ReLU: 3-2                    --
│    │    └─Linear: 3-3                  9,312
│    └─Linear: 2-3                       36
├─BatchNorm: 1-2                         --
│    └─BatchNorm1d: 2-4                  192
├─GINEConv: 1-3                          --
│    └─SumAggregation: 2-5               --
│    └─Sequential: 2-6                   --
│    │    └─Linear: 3-4                  9,312
│    │    └─ReLU: 3-5                    --
│    │    └─Linear: 3-6                  9,312
│    └─Linear: 2-7                       384
├─BatchNorm: 1-4                         --
│    └─BatchNorm1d: 2-8                  192
├─GINEConv: 1-5                          --
│    └─SumAggregation: 2-9               --
│    └─Sequent

In [32]:
results_gru_dummy_descs = train_cls(model = model_gru_dummy_descs,
    optimizer = optimizer_gru_dummy_descs,
    loss_function = LOSS_FUNCTION,
    train_loader = train_loader,
    val_loader = valid_loader,
    num_epochs = EPOCHS,
    device = device,
    edge_attr = True,
    pass_data = True,
    tensorboard_writer = "model_gru_dummy_descs")

Iteration:   0%|          | 0/52 [00:00<?, ?it/s]

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch: 001, Train loss: 0.4836, Train ROC-AUC: 0.7406, Val loss: 0.4059, Val ROC-AUC: 0.8955


Iteration:   0%|          | 0/52 [00:00<?, ?it/s]

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

...

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch: 049, Train loss: 0.2878, Train ROC-AUC: 0.9285, Val loss: 0.3332, Val ROC-AUC: 0.9300


Iteration:   0%|          | 0/52 [00:00<?, ?it/s]

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch: 050, Train loss: 0.2175, Train ROC-AUC: 0.9591, Val loss: 0.3312, Val ROC-AUC: 0.9300


- ### model_gru_nodummy_both

In [33]:
model_gru_nodummy_both = GINGAT(node_dim=9,
                               edge_dim=3,
                               hidden_channels=96,
                               out_channels=N_COMPONENTS,
                               heads=6, dropout=0.2,
                               pooling_type='gru',
                               num_tasks=1,
                               use_dummy=False,
                               feature_mode='both')

optimizer_gru_nodummy_both = torch.optim.Adam(model_gru_nodummy_both.parameters(), lr=0.001, weight_decay=0.0001)

summary(model_gru_nodummy_both)

Layer (type:depth-idx)                   Param #
GINGAT                                   --
├─GINEConv: 1-1                          --
│    └─SumAggregation: 2-1               --
│    └─Sequential: 2-2                   --
│    │    └─Linear: 3-1                  960
│    │    └─ReLU: 3-2                    --
│    │    └─Linear: 3-3                  9,312
│    └─Linear: 2-3                       36
├─BatchNorm: 1-2                         --
│    └─BatchNorm1d: 2-4                  192
├─GINEConv: 1-3                          --
│    └─SumAggregation: 2-5               --
│    └─Sequential: 2-6                   --
│    │    └─Linear: 3-4                  9,312
│    │    └─ReLU: 3-5                    --
│    │    └─Linear: 3-6                  9,312
│    └─Linear: 2-7                       384
├─BatchNorm: 1-4                         --
│    └─BatchNorm1d: 2-8                  192
├─GINEConv: 1-5                          --
│    └─SumAggregation: 2-9               --
│    └─Sequent

In [34]:
results_gru_nodummy_both = train_cls(model = model_gru_nodummy_both,
    optimizer = optimizer_gru_nodummy_both,
    loss_function = LOSS_FUNCTION,
    train_loader = train_loader,
    val_loader = valid_loader,
    num_epochs = EPOCHS,
    device = device,
    edge_attr = True,
    pass_data = True,
    tensorboard_writer = "model_gru_nodummy_both")

Iteration:   0%|          | 0/52 [00:00<?, ?it/s]

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch: 001, Train loss: 0.5645, Train ROC-AUC: 0.6250, Val loss: 0.4002, Val ROC-AUC: 0.8335


Iteration:   0%|          | 0/52 [00:00<?, ?it/s]

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

...

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch: 049, Train loss: 0.0909, Train ROC-AUC: 0.9932, Val loss: 0.3284, Val ROC-AUC: 0.9434


Iteration:   0%|          | 0/52 [00:00<?, ?it/s]

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch: 050, Train loss: 0.0911, Train ROC-AUC: 0.9930, Val loss: 0.3110, Val ROC-AUC: 0.9439


## Test Results

In [37]:
from modules.data_handler import scaffold_split_indices, FingerprintsDescriptorsCalculator, PCAReducer, DTsetBasic
from torch_geometric.loader import DataLoader

## For Test Result :
import os
import numpy as np
import torch
import torch.nn as nn
from torch import device
from torch.utils.data import DataLoader
from torch.nn import Linear
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

from torch_geometric.nn import GINConv
from torch_geometric.nn import global_add_pool
from torch_geometric.loader import DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from torch.optim import Adam

from torch_geometric.nn import GCNConv, TopKPooling, global_mean_pool
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp

from copy import deepcopy
from math import sqrt
from tqdm.notebook import tqdm

In [38]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

- ### results_gru_dummy_both

In [41]:
results_gru_dummy_both

{'best_model': GINGAT(
   (graph_conv1): GINEConv(nn=Sequential(
     (0): Linear(in_features=9, out_features=64, bias=True)
     (1): ReLU()
     (2): Linear(in_features=64, out_features=64, bias=True)
   ))
   (graph_bn1): BatchNorm(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (graph_conv2): GINEConv(nn=Sequential(
     (0): Linear(in_features=64, out_features=64, bias=True)
     (1): ReLU()
     (2): Linear(in_features=64, out_features=64, bias=True)
   ))
   (graph_bn2): BatchNorm(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (graph_conv3): GINEConv(nn=Sequential(
     (0): Linear(in_features=64, out_features=64, bias=True)
     (1): ReLU()
     (2): Linear(in_features=64, out_features=64, bias=True)
   ))
   (graph_bn3): BatchNorm(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (graph_conv4): GINEConv(nn=Sequential(
     (0): Linear(in_features=64, out_features=64, bias=True)
     (1): ReLU()
     (2): Linea

In [42]:
best_gru_dummy_both = results_gru_dummy_both['best_model']

_ , test_auc_gru_dummy_both = run_epoch_cls(model = best_gru_dummy_both, optimizer=None, data_loader=test_loader,
    loss_function=LOSS_FUNCTION, device='cuda', edge_attr=True, pass_data=True)

print(f"Test Result :  AUC: {test_auc_gru_dummy_both:.4f}")

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

Test Result :  AUC: 0.9048


- ### results_mean_dummy_both

In [43]:
results_mean_dummy_both

{'best_model': GINGAT(
   (graph_conv1): GINEConv(nn=Sequential(
     (0): Linear(in_features=9, out_features=96, bias=True)
     (1): ReLU()
     (2): Linear(in_features=96, out_features=96, bias=True)
   ))
   (graph_bn1): BatchNorm(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (graph_conv2): GINEConv(nn=Sequential(
     (0): Linear(in_features=96, out_features=96, bias=True)
     (1): ReLU()
     (2): Linear(in_features=96, out_features=96, bias=True)
   ))
   (graph_bn2): BatchNorm(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (graph_conv3): GINEConv(nn=Sequential(
     (0): Linear(in_features=96, out_features=96, bias=True)
     (1): ReLU()
     (2): Linear(in_features=96, out_features=96, bias=True)
   ))
   (graph_bn3): BatchNorm(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (graph_conv4): GINEConv(nn=Sequential(
     (0): Linear(in_features=96, out_features=64, bias=True)
     (1): ReLU()
     (2): Linea

In [44]:
best_mean_dummy_both = results_mean_dummy_both['best_model']

_ , test_auc_mean_dummy_both = run_epoch_cls(model = best_mean_dummy_both, optimizer=None, data_loader=test_loader,
    loss_function=LOSS_FUNCTION, device='cuda', edge_attr=True, pass_data=True)

print(f"Test Result :  AUC: {test_auc_mean_dummy_both:.4f}")

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

Test Result :  AUC: 0.9359


- ### results_gru_dummy_fps

In [45]:
results_gru_dummy_fps

{'best_model': GINGAT(
   (graph_conv1): GINEConv(nn=Sequential(
     (0): Linear(in_features=9, out_features=96, bias=True)
     (1): ReLU()
     (2): Linear(in_features=96, out_features=96, bias=True)
   ))
   (graph_bn1): BatchNorm(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (graph_conv2): GINEConv(nn=Sequential(
     (0): Linear(in_features=96, out_features=96, bias=True)
     (1): ReLU()
     (2): Linear(in_features=96, out_features=96, bias=True)
   ))
   (graph_bn2): BatchNorm(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (graph_conv3): GINEConv(nn=Sequential(
     (0): Linear(in_features=96, out_features=96, bias=True)
     (1): ReLU()
     (2): Linear(in_features=96, out_features=96, bias=True)
   ))
   (graph_bn3): BatchNorm(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (graph_conv4): GINEConv(nn=Sequential(
     (0): Linear(in_features=96, out_features=64, bias=True)
     (1): ReLU()
     (2): Linea

In [46]:
best_gru_dummy_fps = results_gru_dummy_fps['best_model']

_ , test_auc_gru_dummy_fps = run_epoch_cls(model = best_gru_dummy_fps, optimizer=None, data_loader=test_loader,
    loss_function=LOSS_FUNCTION, device='cuda', edge_attr=True, pass_data=True)

print(f"Test Result :  AUC: {test_auc_gru_dummy_fps:.4f}")

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

Test Result :  AUC: 0.9199


- ### results_gru_dummy_descs

In [47]:
results_gru_dummy_descs

{'best_model': GINGAT(
   (graph_conv1): GINEConv(nn=Sequential(
     (0): Linear(in_features=9, out_features=96, bias=True)
     (1): ReLU()
     (2): Linear(in_features=96, out_features=96, bias=True)
   ))
   (graph_bn1): BatchNorm(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (graph_conv2): GINEConv(nn=Sequential(
     (0): Linear(in_features=96, out_features=96, bias=True)
     (1): ReLU()
     (2): Linear(in_features=96, out_features=96, bias=True)
   ))
   (graph_bn2): BatchNorm(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (graph_conv3): GINEConv(nn=Sequential(
     (0): Linear(in_features=96, out_features=96, bias=True)
     (1): ReLU()
     (2): Linear(in_features=96, out_features=96, bias=True)
   ))
   (graph_bn3): BatchNorm(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (graph_conv4): GINEConv(nn=Sequential(
     (0): Linear(in_features=96, out_features=64, bias=True)
     (1): ReLU()
     (2): Linea

In [48]:
best_gru_dummy_descs= results_gru_dummy_descs['best_model']

_ , test_auc_gru_dummy_descs = run_epoch_cls(model = best_gru_dummy_descs, optimizer=None, data_loader=test_loader,
    loss_function=LOSS_FUNCTION, device='cuda', edge_attr=True, pass_data=True)

print(f"Test Result :  AUC: {test_auc_gru_dummy_descs:.4f}")

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

Test Result :  AUC: 0.8789


- ### results_gru_nodummy_both

In [49]:
results_gru_nodummy_both

{'best_model': GINGAT(
   (graph_conv1): GINEConv(nn=Sequential(
     (0): Linear(in_features=9, out_features=96, bias=True)
     (1): ReLU()
     (2): Linear(in_features=96, out_features=96, bias=True)
   ))
   (graph_bn1): BatchNorm(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (graph_conv2): GINEConv(nn=Sequential(
     (0): Linear(in_features=96, out_features=96, bias=True)
     (1): ReLU()
     (2): Linear(in_features=96, out_features=96, bias=True)
   ))
   (graph_bn2): BatchNorm(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (graph_conv3): GINEConv(nn=Sequential(
     (0): Linear(in_features=96, out_features=96, bias=True)
     (1): ReLU()
     (2): Linear(in_features=96, out_features=96, bias=True)
   ))
   (graph_bn3): BatchNorm(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (graph_conv4): GINEConv(nn=Sequential(
     (0): Linear(in_features=96, out_features=64, bias=True)
     (1): ReLU()
     (2): Linea

In [50]:
best_gru_nodummy_both = results_gru_nodummy_both['best_model']

_ , test_auc_gru_nodummy_both = run_epoch_cls(model = best_gru_nodummy_both, optimizer=None, data_loader=test_loader,
    loss_function=LOSS_FUNCTION, device='cuda', edge_attr=True, pass_data=True)

print(f"Test Result :  AUC: {test_auc_gru_nodummy_both:.4f}")

Iteration:   0%|          | 0/7 [00:00<?, ?it/s]

Test Result :  AUC: 0.9179
